In [ ]:
from pathlib import Path
import sys
from datetime import date
import pandas as pd
import gc  
import os
import glob
import numpy as np
import pickle

# --- Paths / imports -------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
PREPROCESSING_DIR = PROJECT_ROOT / "functions" / "preprocessing"
for p in (PROJECT_ROOT, PREPROCESSING_DIR):
    if str(p) not in sys.path:
        sys.path.append(str(p))

from server_config import datapath, proj_sheet, preprocessed_path, raw_path, backup_path, preprocessed_path_freezed
from missing_data import compute_availability_metrics

# --- Dates ------------------------------------------------------------
today_str = date.today().strftime("%d%m%Y")        
today_day = pd.Timestamp.today().normalize()       
today_str = "25082025"

# --- Path -------------------------------------------------------------

datapath = Path(raw_path) / f"export_tiki_{today_str}"  

In [ ]:
# actual passive + ema_data
file_pattern = os.path.join(datapath, "epoch_part*.csv")
file_list = glob.glob(file_pattern)
file_list.sort()
df_complete = pd.concat((pd.read_csv(f, encoding="latin-1", low_memory=False) for f in file_list), ignore_index=True)

In [ ]:
for col in ["startTimestamp", "endTimestamp", "createdAt"]:
    df_complete[col] = pd.to_datetime(
        df_complete[col], utc=True, errors="coerce", unit="ms"
    )


In [ ]:
print(df_complete["startTimestamp"].min(), df_complete["startTimestamp"].max())
print(df_complete["endTimestamp"].min(), df_complete["endTimestamp"].max())
print(df_complete["createdAt"].min(), df_complete["createdAt"].max())

In [ ]:
df_complete["type"].unique()

In [ ]:
# # Merge with backup data
# backup_path = preprocessed_path + "/backup_passive_05052025.parquet"
# # backup_path = preprocessed_path + "/backup_passive_last.feather"
# df_backup = pd.read_parquet(backup_path)

In [ ]:
df_complete["type"].unique()

In [ ]:
df_complete["start_day"] = df_complete["startTimestamp"].dt.floor("D")
df_complete["end_day"] = df_complete["endTimestamp"].dt.floor("D")
df_complete["created_day"] = df_complete["createdAt"].dt.floor("D")


In [ ]:
df_complete

In [ ]:
# Check column names to identify customer and timezoneOffset columns
print("Columns in df_complete:")
print(df_complete.columns.tolist())
print(f"\nDataFrame shape: {df_complete.shape}")

# Look for columns that might contain customer/user information
customer_cols = [col for col in df_complete.columns if any(term in col.lower() for term in ['user', 'customer', 'id', 'participant'])]
print(f"\nPotential customer columns: {customer_cols}")

# Look for timezone-related columns
timezone_cols = [col for col in df_complete.columns if any(term in col.lower() for term in ['timezone', 'offset', 'time'])]
print(f"\nTimezone-related columns: {timezone_cols}")

In [ ]:
# Create a series with MultiIndex (customer, start_day) and unique timezoneOffset values
timezone_series = df_complete.groupby(['customer', 'start_day'])['timezoneOffset'].apply(
    lambda x: x.dropna().unique()
).reset_index()

# Convert to a proper series with MultiIndex
timezone_series_indexed = timezone_series.set_index(['customer', 'start_day'])['timezoneOffset']

print(f"Series shape: {timezone_series_indexed.shape}")
print(f"Number of unique customer-start_day combinations: {len(timezone_series_indexed)}")
print("\nFirst 10 entries:")
print(timezone_series_indexed.head(10))

In [ ]:
# Analyze the timezone offsets
print("Analysis of timezoneOffset patterns:")
print("="*50)

# Check how many unique timezone offsets each customer-day combination has
offset_counts = timezone_series_indexed.apply(len)
print(f"Distribution of number of unique offsets per customer-day:")
print(offset_counts.value_counts().sort_index())

# Find cases with multiple timezone offsets on the same day
multiple_offsets = timezone_series_indexed[offset_counts > 1]
print(f"\nCustomer-day combinations with multiple timezone offsets: {len(multiple_offsets)}")

if len(multiple_offsets) > 0:
    print("\nFirst few examples of multiple offsets per day:")
    for i, (idx, offsets) in enumerate(multiple_offsets.head(5).items()):
        customer, day = idx
        print(f"  {customer} on {day.date()}: {offsets}")

# Show some statistics about timezone offsets
all_offsets = df_complete['timezoneOffset'].dropna()
print(f"\nOverall timezone offset statistics:")
print(f"  Total non-null timezone offset records: {len(all_offsets):,}")
print(f"  Unique timezone offset values: {sorted(all_offsets.unique())}")
print(f"  Most common offset: {all_offsets.mode().iloc[0]} (appears {(all_offsets == all_offsets.mode().iloc[0]).sum():,} times)")

# Store the final series for easy access
print(f"\nFinal series 'timezone_series_indexed' created with:")
print(f"  - MultiIndex: (customer, start_day)")  
print(f"  - Values: unique timezoneOffset arrays for each customer-day")
print(f"  - Shape: {timezone_series_indexed.shape}")

In [ ]:
c, d = timezone_series_indexed[offset_counts >= 3].index[2]
# df_complete[(df_complete["customer"] == c) & (df_complete["start_day"] == d)].groupby("type")["timezoneOffset"].unique()
df_cd = df_complete[(df_complete["customer"] == c) & (df_complete["start_day"] == d)]
df_cd
df_cd["generation"].apply(lambda x: generation_type_map[x])

# generatioin & trustworthiness

generation:

| Parameter | Description |
|-----------|-------------|
| manual_entry | Data was manually entered by a user |
| manual_measurement | Data was recorded by a sensor measurement manually triggered by a user |
| automated_measurement | Data was recorded by a passive sensor measurement |
| ~~smartphone~~ | Data was recorded by a smartphone sensor |
| tracker | Data was recorded by a wearable sensor or medical device |
| third_party | Data was recorded by a third-party app |
| calculation | Data was calculated by Thryve (e.g., daily steps if not provided by manufacturer)


trustworthiness:

| Parameter | Description |
|-----------|-------------|
| **_unfavorable_measurement_context_** | Device manufacturer believes recording was made under suboptimal conditions (e.g., during movement when stillness is recommended) |
| **_doubt_from_device_source_** | Device algorithms flagged the measurement as potentially unreliable |
| ~~doubt_from_user~~ | User manually tagged the recording as unlikely or implausible |
| **_verified_from_device_source_** | Device manufacturer considers the recording plausible and reliable |
| ~~verified_from_user~~ | User manually verified |



In [ ]:
print("generation:", df_complete.generation.unique())
print("trustworthiness:", df_complete.trustworthiness.unique())
print("medicalGrade:", df_complete.medicalGrade.unique())
print("userReliability:", df_complete.userReliability.unique())
print("chronologicalExactness:", df_complete.chronologicalExactness.unique())

In [ ]:
generation_type_map = {
    -1: "unknown",
    10: "manual_entry",
    20: "manual_measurement",
    30: "automated_measurement",
    40: "calculation",
    50: "smartphone", # not present
    60: "tracker",
    70: "third_party"
}

trustworthiness_type_map = {
    -1: "unknown",
    10: "plausible",
    20: "verified_from_device_source", # present
    30: "verified_from_user",
    40: "verified_from_external_source",
    50: "unlikely",
    60: "implausible",
    70: "unfavorable_measurement_context", # present
    80: "insufficient_database",
    90: "doubt_from_device_source", # present
    100: "doubt_from_user"
}

In [ ]:
[generation_type_map[int(k)] for k in sorted(df_complete.generation.dropna().unique())]

In [ ]:
[trustworthiness_type_map[int(k)] for k in sorted(df_complete.trustworthiness.dropna().unique())]

In [ ]:
result = df_complete.groupby("generation")["type"].unique()
for generation, types in result.items():
    generation_name = generation_type_map.get(generation, f"Unknown ({generation})")
    print(f"{generation_name} ({generation}):")
    for type_name in types:
        print(f"  - {type_name}")
    print()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Create a 2D histogram of generation vs type, normalized by type, including NA values

# First, let's handle NaN values by replacing them with a specific label
df_plot = df_complete.copy()
df_plot['generation_labeled'] = df_plot['generation'].fillna(-1)  # Use -1 for NA values
df_plot['type_clean'] = df_plot['type'].fillna('NA')

# Map generation values to readable labels
generation_labels = generation_type_map.copy()
generation_labels[-1] = 'NA'  # Add NA label

# Create the cross-tabulation for the heatmap
crosstab = pd.crosstab(df_plot['type_clean'], df_plot['generation_labeled'], normalize='index')

# Map column names to readable labels
crosstab.columns = [generation_labels.get(col, f'Unknown_{col}') for col in crosstab.columns]

# Convert to percentages for display
crosstab_percent = crosstab * 100

# Create the plot with percentages
plt.figure(figsize=(12, 8))
sns.heatmap(crosstab_percent, annot=True, fmt='.1f', cmap='inferno_r', 
            cbar_kws={'label': 'Percentage (normalized by type)'})

plt.title('2D Histogram: Density of Generation Types by Data Type\n(Normalized by Type, Including NA values)')
plt.xlabel('Generation Type')
plt.ylabel('Data Type')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Also create a regular count heatmap for reference
plt.figure(figsize=(12, 8))
crosstab_counts = pd.crosstab(df_plot['type_clean'], df_plot['generation_labeled'])
crosstab_counts.columns = [generation_labels.get(col, f'Unknown_{col}') for col in crosstab_counts.columns]

sns.heatmap(crosstab_counts, annot=True, fmt='_d', cmap='viridis_r', 
            cbar_kws={'label': 'Count'})

plt.title('2D Histogram: Count of Generation Types by Data Type\n(Raw Counts, Including NA values)')
plt.xlabel('Generation Type')
plt.ylabel('Data Type')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Print some summary statistics
print("Summary of Generation Types by Data Type (Percentages):")
print(crosstab.round(1))
print(f"\nTotal records: {len(df_complete):,}")
print(f"Records with NA generation: {df_complete['generation'].isna().sum():,}")
print(f"Records with NA type: {df_complete['type'].isna().sum():,}")

# 2. Monitoring data

In [ ]:
df_monitoring = pd.read_csv(f"https://docs.google.com/spreadsheets/d/{proj_sheet}/export?format=csv")

In [ ]:
df_monitoring

In [ ]:
session        = pd.read_csv(datapath / "questionnaireSession.csv", low_memory=False)
answers        = pd.read_csv(datapath / "answers.csv", low_memory=False)
choice         = pd.read_csv(datapath / "choice.csv", low_memory=False)
questions      = pd.read_csv(datapath / "questions.csv", low_memory=False)
questionnaire  = pd.read_csv(datapath / "questionnaires.csv", low_memory=False)  # ohne Komma!


In [ ]:
session

In [ ]:
df_complete

In [ ]:
for col in ["startTimestamp", "endTimestamp", "createdAt"]:
    df_complete[col] = pd.to_datetime(df_complete[col], errors="coerce", utc=True)

In [ ]:
df_complete

In [ ]:
df_complete